# Notebook 4: watching a trained agent play

After [Worksheet 3](03_train_step.ipynb), you can turn a probability
into a sampled action, score it, and push weights around. Everything so far ran on fake tensors.

This notebook connects those pieces to the real thing:

1. load a checkpoint trained by `scripts/train.py`
2. turn a real Atari frame into the 6400-number input vector
3. take one decision with the trained policy
4. watch a full episode, inline, with the policy's own probability shown beside it

There is still no training here. This is inference, the weights are frozen.

Use the `pong-pytorch` kernel.

## Running this notebook

Locally: the environment comes from the repo. `uv sync --locked`, then
`uv run jupyter lab`. The cell below does nothing.

On Colab: the kernel exists before the project does, so the cell below clones the
repo and installs what Colab is missing. Run it first.

It installs `gymnasium` and `ale-py` only. Colab already provides a working CUDA build
of torch, and replacing it with the exact locked one means downloading the entire CUDA
stack for no benefit to anything taught here. The README describes the stricter route
if you ever need bit-identical versions.

The cell ends by importing `pong`, so if anything above failed you find out here rather
than three cells later.


In [ ]:
# Colab bootstrap. Does nothing when you run this notebook locally.
import sys

IN_COLAB = "google.colab" in sys.modules
REPO = "/content/pong-pytorch"

if IN_COLAB:
    # The -d test makes re-running this cell harmless.
    ![ -d {REPO} ] || git clone -q https://github.com/piyushahuja/pong-pytorch.git {REPO}
    %cd {REPO}
    !curl -LsSf https://astral.sh/uv/install.sh | env UV_INSTALL_DIR=/usr/local/bin sh
    !uv pip install -q --system --python {sys.executable} gymnasium ale-py

    # Put the package on the path directly rather than installing it editable.
    # An editable install works by dropping a .pth file into site-packages, and
    # .pth files are only read when the interpreter starts, so it would not take
    # effect in this kernel, which is already running.
    sys.path.insert(0, f"{REPO}/src")

import pong        # fails here, loudly, if any of the above did not work

print("python:", sys.version.split()[0])
print("pong package from:", pong.PROJECT_ROOT)


In [ ]:
# Confirm the environment is what you think it is.
import torch
import gymnasium
import ale_py

print("torch:     ", torch.__version__)
print("gymnasium: ", gymnasium.__version__)
print("ale-py:    ", ale_py.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import torch
import matplotlib.pyplot as plt

import pong        # Policy, preprocess, make_env, rollout
from pong import replay        # watch, animate

print(torch.__version__)

## What a checkpoint actually holds

`scripts/train.py` saves a dictionary, not a bare model. Four keys:

| Key | What it is | Needed to play? |
|---|---|---|
| `model_state_dict` | the learned weights | yes |
| `optimizer_state_dict` | RMSprop's running averages | no, only to *resume training* |
| `episode` | how many episodes produced it | no |
| `running_reward` | training-time score, smoothed | no |

A `state_dict` is just an ordered mapping from parameter name to tensor. It carries no
architecture, which is why you must construct the same `Policy` class before loading into it.

In [ ]:
# pong.default_checkpoint() prefers a live pong_policy.pt from a local
# training run, and otherwise falls back to the newest archived policy in
# checkpoints/, which is what a fresh clone has.
path = pong.default_checkpoint()
print('checkpoint:', path)

ckpt = torch.load(path, map_location='cpu', weights_only=False)

print('keys:', list(ckpt))
print('trained for', ckpt['episode'], 'episodes')
print('training running reward:', round(ckpt['running_reward'], 2))
print()
for name, tensor in ckpt['model_state_dict'].items():
    print(f'{name:12s} {tuple(tensor.shape)}  {tensor.numel():,} parameters')

That is the same two-layer shape as `PongShapedNet` in
[Notebook 2](02_neural_network.ipynb), $6400 \rightarrow 200 \rightarrow 1$, with one
difference: no biases, matching the original NumPy implementation.

$$
p = \sigma\!\left(W_2 \;\mathrm{relu}(W_1 x)\right), \qquad
W_1 \in \mathbb{R}^{200 \times 6400}, \quad W_2 \in \mathbb{R}^{1 \times 200}
$$

Two matrices. That is the entire agent.

In [ ]:
policy = pong.load_policy()
policy.eval()

total = sum(p.numel() for p in policy.parameters())
print(policy)
print(f'\ntotal parameters: {total:,}')

## From pixels to 6400 numbers

ALE hands you a `210 x 160 x 3` RGB frame. The policy wants a flat vector. `pong.preprocess`
does four things, in order:

| Step | Code | Shape after |
|---|---|---|
| Crop score bar and floor | `x[35:195]` | `160 x 160 x 3` |
| Downsample, keep one channel | `x[::2, ::2, 0]` | `80 x 80` |
| Background to 0, everything else to 1 | `(x != 144) & (x != 109) & (x != 0)` | `80 x 80` |
| Flatten | `x.flatten()` | `6400` |

Note what survives: two paddles and a ball, as ones on a field of zeros. Colour, score
digits and court markings are all discarded. The agent never sees them.

In [ ]:
env = pong.make_env('rgb_array')
observation, _ = env.reset(seed=1)

# Step a little so the ball is actually in play.
for _ in range(22):
    observation, *_ = env.step(0)

print('raw frame from ALE:', observation.shape, observation.dtype)

In [ ]:
# The same four steps, kept separate so you can see each one.
raw       = torch.from_numpy(observation)
cropped   = raw[35:195]
downsized = cropped[::2, ::2, 0]
binary    = ((downsized != 144) & (downsized != 109) & (downsized != 0)).float()
flat      = binary.flatten()

stages = [
    (raw,       f'raw  {tuple(raw.shape)}'),
    (cropped,   f'crop  {tuple(cropped.shape)}'),
    (downsized, f'downsample  {tuple(downsized.shape)}'),
    (binary,    f'binarise  {tuple(binary.shape)}'),
]

fig, axes = plt.subplots(1, 4, figsize=(11, 3.2))
fig.patch.set_facecolor('#f4f7f6')
for ax, (img, label) in zip(axes, stages):
    ax.imshow(img.numpy(), cmap='gray' if img.ndim == 2 else None)
    ax.set_title(label, fontsize=9, color='#1a2b2e')
    ax.axis('off')
plt.tight_layout()
plt.show()

print('flattened:', tuple(flat.shape))
print('fraction of pixels that are 1:', round(flat.mean().item(), 4))

## The network sees motion, not a still

A single binarised frame is ambiguous: it shows *where* the ball is, never *where it is going*.
A policy fed one frame cannot tell an approaching ball from a departing one.

The fix costs nothing. Feed the difference of consecutive frames:

$$
x_t = \text{prepro}(o_t) - \text{prepro}(o_{t-1})
$$

Now moving objects appear twice, $+1$ where they arrived, $-1$ where they left, and static
objects cancel to zero. Direction is encoded in the sign. On the very first step there is no
previous frame, so `scripts/train.py` passes a vector of zeros.

In [ ]:
previous = pong.preprocess(observation)
observation, *_ = env.step(0)
current  = pong.preprocess(observation)

difference = current - previous

fig, axes = plt.subplots(1, 3, figsize=(8.5, 3.2))
fig.patch.set_facecolor('#f4f7f6')
for ax, (img, label, cmap) in zip(axes, [
    (previous,   'previous frame',  'gray'),
    (current,    'current frame',   'gray'),
    (difference, 'difference (the actual input)', 'bwr'),
]):
    ax.imshow(img.reshape(80, 80).numpy(), cmap=cmap, vmin=-1, vmax=1)
    ax.set_title(label, fontsize=9, color='#1a2b2e')
    ax.axis('off')
plt.tight_layout()
plt.show()

print('nonzero in a single frame :', int((current != 0).sum()))
print('nonzero in the difference :', int((difference != 0).sum()))
print('value range               :', difference.min().item(), 'to', difference.max().item())

## One decision

This is [Notebook 3](03_train_step.ipynb)'s Bernoulli section, with a real $p$ from real
pixels. The network emits one number; ALE wants an integer action code:

| Bernoulli sample | Atari action | Meaning |
|---|---|---|
| `1` | `2` | UP |
| `0` | `3` | DOWN |

Only two of Pong's actions are ever used. The agent cannot stay still, and does not need to.

In [ ]:
with torch.no_grad():
    p = policy(difference)

dist = torch.distributions.Bernoulli(probs=p)
sample = dist.sample()

print(f'P(UP)          : {p.item():.4f}')
print(f'sampled        : {int(sample.item())}  -> Atari action {2 if sample.item() == 1 else 3}')
print(f'log_prob       : {dist.log_prob(sample).item():.4f}')
print(f'greedy (argmax): {2 if p.item() > 0.5 else 3}')

## Greedy or sampling? The exploration tax

During training the agent must sample. Sampling is what produces the variation that
`log_prob` can then be pushed up or down, a deterministic policy generates no learning signal.

At play time that constraint is gone. You can take the argmax and keep the noise.

The `running_reward` printed above was measured while sampling, during training.
It therefore understates what the policy can do once it stops exploring. Compare the two
read-outs yourself:

In [ ]:
results = []
print('playing 6 full episodes, this takes ~30s...')
for seed in range(3):
    for greedy in (True, False):
        us, them, _, _ = pong.rollout(policy, env, greedy=greedy, seed=seed, verbose=False)
        results.append((seed, 'argmax' if greedy else 'sample', us, them))

print(f"\n{'seed':>5}  {'mode':>7}  {'score':>7}  result")
for seed, mode, us, them in results:
    print(f'{seed:>5}  {mode:>7}  {us:>3} - {them:<3}  {"WIN" if us > them else "loss"}')

Same weights, two read-outs. Any gap between them is the price of exploring, and it is
refundable the moment learning stops, but with a well-trained policy expect the gap to be
small or absent. Sampling and argmax only disagree when the policy is unsure, and this one
is confident on most frames. Note too that the seed-to-seed spread is wide, so three seeds
cannot establish a difference either way.

## Watch it play

`replay.watch` runs one episode with `render_mode='rgb_array'`, collects the frames, and returns
an HTML5 player. The bar on the right is the policy's own $P(\text{UP})$ at that frame, teal
above $0.5$, red below. Scrub it and watch the bar commit as the ball approaches.

Size warning. Each frame costs roughly 13 KB of embedded PNG. `max_steps=1500` gives five or
six points for about 4 MB. A full 21-point episode would bloat the notebook to ~17 MB, raise
`frame_stride` if you want the whole match.

In [ ]:
replay.watch(seed=1, max_steps=1500, frame_stride=3)

## The live window

The player above is a recording. For a real-time pygame window, ALE needs to own an OS window,
which a notebook cannot host. Run it from a terminal instead:

```bash
uv run scripts/play.py --mode human --seed 1            # argmax
uv run scripts/play.py --mode human --seed 1 --sample   # as trained
uv run scripts/play.py --mode human --episodes 5 --sticky 0.0
```

`--sticky` controls `repeat_action_probability`. The default `0.25` reproduces training
conditions (and the old `Pong-v0` default); `0.0` removes the action-repeat noise and the agent
looks sharper, but it is no longer the distribution it was trained on.

In [ ]:
env.close()

## What you can do now

| Skill | API |
|---|---|
| Restore weights | `Policy(); load_state_dict(ckpt['model_state_dict'])` |
| Pixels to features | `pong.preprocess(observation)` |
| Encode motion | `preprocess(o_t) - preprocess(o_{t-1})` |
| Act without learning | `torch.no_grad()` + argmax |
| Watch inline | `replay.watch(seed=...)` |

Still ahead: the training loop itself, running this forward pass thousands of times per
episode, collecting `log_prob` and reward at every step, discounting across rally boundaries, and
accumulating gradients over ten episodes before each `optimizer.step()`. That is
`scripts/train.py`, read it top to bottom; it is written in this order.